In [1]:
import pandas as pd

caminho_arquivo = "dados_saneamento/fortaleza_dados_saneamento.xlsx"

# lê o excel
df_saneamento_dados = pd.read_excel(
    caminho_arquivo,
    header=2
)

# pega as linhas desejadas
df_saneamento_formatado = df_saneamento_dados.iloc[29:37]

# transpõe a tabela
df_saneamento_formatado = df_saneamento_formatado.T

# primeira linha vira cabeçalho
df_saneamento_formatado.columns = (
    df_saneamento_formatado.iloc[0]
)

# remove linha usada no cabeçalho
df_saneamento_formatado = (
    df_saneamento_formatado.iloc[1:]
    .reset_index()
)

df_saneamento_formatado = (
    df_saneamento_formatado.rename(
        columns={
            "index": "Ano",
            "População total que mora em domicílios com acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_total_com_coleta_esgoto",
            "População total que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_sem_coleta_esgoto",
            "Parcela da população total que mora em domicílios com acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_total_com_coleta_esgoto",
            "Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_sem_coleta_esgoto",
            "População urbana que mora em domicílios com acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_urbana_com_coleta_esgoto",
            "População urbana que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA)": "Pop_urbana_sem_coleta_esgoto",
            "Parcela da população urbana que mora em domicílios com acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_urbana_com_coleta_esgoto",
            "Parcela da população urbana que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA)": "Perc_pop_urbana_sem_coleta_esgoto",               
        }
    )
)

colunas_percentuais = [
    "Perc_pop_total_com_coleta_esgoto",
    "Perc_sem_coleta_esgoto",
    "Perc_pop_urbana_com_coleta_esgoto",
    "Perc_pop_urbana_sem_coleta_esgoto"
]

print(
    df_saneamento_formatado[
        colunas_percentuais
    ].head()
)


for coluna in colunas_percentuais:
    df_saneamento_formatado[coluna] = (
        pd.to_numeric(
            df_saneamento_formatado[coluna],
            errors="coerce"
        ).round(4)
    )


# substituir - por nao_informado
df_saneamento_formatado = (
   df_saneamento_formatado
    .replace("-", "nao_informado")
)

# drop nas colunas irrelevantes
colunas_drop = [
    "Pop_total_com_coleta_esgoto",
    "Perc_pop_total_com_coleta_esgoto",
    "Perc_pop_urbana_com_coleta_esgoto",
    "Perc_pop_urbana_sem_coleta_esgoto",
    "Pop_urbana_com_coleta_esgoto",
    "Pop_urbana_sem_coleta_esgoto",
]

df_saneamento_formatado = df_saneamento_formatado.drop(
    columns=colunas_drop
)

meses = range(1, 13)

novas_linhas = []

for _, linha in df_saneamento_formatado.iterrows():
    for mes in meses:
        novas_linhas.append({
            "MES_REFERENCIA": f"{linha['Ano']}-{mes:02d}",
            "Pop_sem_coleta_esgoto": linha["Pop_sem_coleta_esgoto"],
            "Perc_sem_coleta_esgoto": linha["Perc_sem_coleta_esgoto"]
        })

df_saneamento_mensal  = pd.DataFrame(novas_linhas)

"""
Ano e mes,
População total que mora em domicílios sem acesso ao serviço de coleta de esgoto (pessoas) (SNIS/SINISA),
Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SNIS/SINISA),
"""

print(df_saneamento_formatado.isnull().sum())

print("Arquivo CSV salvo com sucesso!")


Indicador Perc_pop_total_com_coleta_esgoto Perc_sem_coleta_esgoto  \
0                                    0.642                  0.358   
1                                    0.665                  0.335   
2                                    0.628                  0.372   
3                                    0.559                  0.441   
4                                    0.553                  0.447   

Indicador Perc_pop_urbana_com_coleta_esgoto Perc_pop_urbana_sem_coleta_esgoto  
0                                     0.642                             0.358  
1                                     0.665                             0.335  
2                                         -                                 -  
3                                     0.559                             0.441  
4                                     0.553                             0.447  
Indicador
Ano                       0
Pop_sem_coleta_esgoto     0
Perc_sem_coleta_esgoto    0
dtype: int6

Pegando somente os dados de 2017 a 2024 e ordenando do menor para o maior

In [2]:
# Pegando os MES_REFERENCIA de 2017 a 2024
datas = pd.date_range(start="2010-01-01", end="2024-12-31", freq="MS")
df_temporal = pd.DataFrame({'MES_REFERENCIA' : datas})
df_temporal['Ano'] = df_temporal['MES_REFERENCIA'].dt.year

df_saneamento_final = pd.merge(
    df_temporal, df_saneamento_formatado[['Ano', 'Pop_sem_coleta_esgoto', 'Perc_sem_coleta_esgoto']], 
    on='Ano', how='left'
    )

# Transformar o 'Ano' em uma data (Primeiro dia do ano: 01/01/XXXX)
df_saneamento_formatado['MES_REFERENCIA'] = pd.to_datetime(df_saneamento_formatado['Ano'].astype(str) + '-01-01')

print(df_saneamento_final.head(15))

#Ordenando por MES_REFERENCIA
df_saneamento_formatado = df_saneamento_formatado.sort_values("MES_REFERENCIA").reset_index(drop=True)

print(df_saneamento_formatado[["MES_REFERENCIA", "Pop_sem_coleta_esgoto", "Perc_sem_coleta_esgoto"]].head())


   MES_REFERENCIA   Ano Pop_sem_coleta_esgoto  Perc_sem_coleta_esgoto
0      2010-01-01  2010               1267675                   0.517
1      2010-02-01  2010               1267675                   0.517
2      2010-03-01  2010               1267675                   0.517
3      2010-04-01  2010               1267675                   0.517
4      2010-05-01  2010               1267675                   0.517
5      2010-06-01  2010               1267675                   0.517
6      2010-07-01  2010               1267675                   0.517
7      2010-08-01  2010               1267675                   0.517
8      2010-09-01  2010               1267675                   0.517
9      2010-10-01  2010               1267675                   0.517
10     2010-11-01  2010               1267675                   0.517
11     2010-12-01  2010               1267675                   0.517
12     2011-01-01  2011               1148295                   0.464
13     2011-02-01  2

Exportando os dados em csv

In [3]:
# salva em csv
df_saneamento_final.to_csv(
    "dados_formatados/SANEAMENTO_DADOS_FORTALEZA.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.3f"
)